# Findings: Product quality

Which product categories are more prone to a bad review, and is that still true when the order arrived on time?

A bad review is a latest score of 1 or 2. The category is the highest-priced item. The rate thresholds are in `config/settings.py`.

## 1. Data quality

Orders by purchase month, then the review and item counts

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import Image, display

root = Path.cwd().resolve()
if not (root / "src").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from config.paths import CATEGORY_CHART
from config.settings import (
    BAD_REVIEW_SCORES,
    MIN_CATEGORY_ORDERS,
    MIN_ON_TIME_GAP,
)
from src.category_rates import chart
from src.load import add_window, attach, run_sql

con = duckdb.connect()
attach(con)
display(run_sql(con, "quality_values.sql"))
display(run_sql(con, "quality_grain.sql"))
display(run_sql(con, "orders_historical.sql"))

,check_name,n
0,null review_score,0
1,null purchase timestamp,0
2,null order_status,0
3,null delivered date,2965
4,null estimated delivery date,0
5,null price,0
6,null product category,610
7,duplicate order_id,0
8,duplicate review_id,789
9,duplicate order item,0


,check_name,n
0,orders,99441
1,orders with more than one review,547
2,review_id used on more than one order,789
3,multi-review orders whose scores disagree,202
4,orders with more than one item,9803
5,orders with no review,768
6,orders with no items,775
7,orders with more than one item,9803
8,products with no category,610
9,category names with no English translation,13


,month,orders
0,2016-09,4
1,2016-10,324
2,2016-11,0
3,2016-12,1
4,2017-01,800
5,2017-02,1780
6,2017-03,2682
7,2017-04,2404
8,2017-05,3700
9,2017-06,3245


#### Conclusion from data analysis

We keep January 2017 through August 2018 (window end 2018-09-01, excluded). Before it, the file has only 329 orders: 4 in 2016-09, 324 in 2016-10, 1 in 2016-12, and none in 2016-11. That zero sits between sparse months, so it is most likely a gap in the file and not a month with no demand. After it, 2018-09 and 2018-10 hold only 16 and 4 orders, so the file ends mid-September. Starting in 2017 and stopping at August 2018 avoids these partial months, which would distort trends.

The data checks are clean on the key fields: no nulls in purchase timestamp, status, estimated delivery, price or review score, no duplicate `order_id`, no scores outside 1 to 5, and no deliveries before purchase.